# Worked solutions · Section 1 of the sample questions

The sample questions are in `documentation/tex/questions`.
For each question the entry gives the answer,
a solution that cites the lecture notes for every formula it uses,
and the misreading that leads to each of the other choices.
The facts that several questions use are stated once, at the head of the section,
as F1–F4.

A citation names the number of a statement in the current build of the notes
and its label, which survives renumbering:
Remark 1.2.4 (`remark.gridAgainstOccupied`), eq. (1.1.2, `eq.queueImbalance`).
Prices are integer counts of ticks, as in the package, unless a question states a tick size.

The code cells are a calculator: arithmetic on the numbers of the question,
as in the examination.
No snippet is run here;
the notebook `01-representing-the-book` runs them and verifies every key.

# Section 1 · Representing the aggregate book

**F1. The book of the package.**
`AggregateBook` holds the configuration of Section 1.1 (`sec.orderDrivenMarkets`)
as two dictionaries from price to size, `bids` and `asks`, Section 1.2.3 (`sec.foldImplementation`).
`size_at` reads the size at a price, zero where nothing rests;
`set_size` writes a size and removes the price when the size is zero,
and every write of the book goes through it;
the best price is `max` of the bids and `min` of the asks,
Listing 1.1 (`listing.aggregateStorage`).
So a price is a key of a side exactly when a positive size rests at it.

**F2. Levels and the derived quantities.**
Levels are counted on the price grid, from 1:
$P^{b,i} = P^b - (i-1)\tau$ and $P^{a,i} = P^a + (i-1)\tau$, with $\tau = 1$ tick here,
and $S^{b,i}$, $S^{a,i}$ are the sizes at those prices,
zero at a grid price where nothing rests
(points iii and iv of the configuration in Section 1.1 (`sec.orderDrivenMarkets`)).
`bid_size(i)` and `ask_size(i)` read them.
When both sides are non-empty,
the definitions that follow Remark 1.1.3 (`remark.sizeVersusVolume`) give
the spread $\phi = |P^a - P^b|$, which is $P^a - P^b$ since the book never crosses,
the mid-price $P^m = (P^a + P^b)/2$
and the queue imbalance, eq. (1.1.2, `eq.queueImbalance`),
$$I^n = \frac{\sum_{i\leq n} S^{b,i} - \sum_{i\leq n} S^{a,i}}{\sum_{i\leq n} S^{b,i} + \sum_{i\leq n} S^{a,i}},$$
positive when the book is bid-heavy.
An aggregated feed reports the first occupied prices of each side instead,
and the two indexings agree only for those $n$ within which no gap falls,
Remark 1.2.4 (`remark.gridAgainstOccupied`).

**F3. Matching.**
Algorithm 1.2.1 (`algo.bookFold`) is the transition of the book.
A cancellation subtracts its size at its price on its side, and removes the price if it reaches zero.
A submission matches against the best price-eligible price of the opposite side,
at that resting price, removing each price it empties,
and rests what is left at its own price.
A market order carries a sentinel price, against which every price of the opposite side is eligible,
and what is left of it rests at the price of its last fill, and nowhere if it matched nothing
(Section 1.2.3 (`sec.foldImplementation`)).
`apply` passes a submission to `submit` and a withdrawal to `withdraw`.

**F4. Dictionaries.**
A fact of Python: a `dict` iterates, and prints, its items in the order in which their keys were first inserted;
assigning to a key already present keeps its place,
and `dict(d)` holds the items of `d` in their order.
A change made to a dictionary through one name is read through every other,
and `dict(d)` is a new dictionary, equal to `d` and not `d` itself,
Section B.1 (`sec.namesAndObjects`) and Listing B.1 (`listing.namesAndObjects`).

## 1.1 · `question.signedStateToSides`

The signed map `{1000: 100, 999: 200, 998: 150, 1001: 0, 1002: -120, 1003: -180}`:
bid sizes positive, ask sizes negative, and 1001 emptied.
A keeps as asks the prices with `s < 0`; B keeps as asks every price that is not a bid.

**Answer: (C).** A prints `1002 2`; B raises `ValueError`.

- **A.** `bids` is `{1000: 100, 999: 200, 998: 150}` and `asks` is `{1002: 120, 1003: 180}`:
  the size 0 at 1001 passes neither test.
  The book has $P^b = 1000$, $P^a = 1002$ and $\phi = 2$ (F2).
- **B.** `bids` is the same, and `asks` takes every other price, 1001 among them, with the size $-0 = 0$.
  `from_levels`, as quoted, refuses a size that is not positive, and raises `ValueError` at 1001.

A price whose queue has emptied is not a key of the book (F1):
A drops it, as `set_size` would.

**The other choices.**
- (A) Both print `1002 2`: takes `from_levels` to pass the zero to `set_size`, which would remove it;
  `from_levels` raises first.
- (B) A prints `1002 2`, B prints `1001 1`: takes the book to hold 1001 as a key of the asks, of size 0,
  the best ask, with $\phi = 1001 - 1000 = 1$.
- (D) Both raise `ValueError`: reads the test `s < 0` of A as `s <= 0`,
  which takes the emptied 1001 into the asks of A as well.

## 1.2 · `question.imbalanceOverGridLevels`

Bids `{1000: 100, 998: 300, 997: 200}` and asks `{1001: 150, 1002: 50, 1004: 400}`:
a gap at 999 on the bid side and at 1003 on the ask side.
`queue_imbalance(GridDepth(2))` against `imbalance(book, 2)`,
which takes the first two occupied prices of each side.

**Answer: (D).** `-0.333 0.333`.

**Over grid levels.**
By F2 the first two bid levels are $P^b = 1000$ and $999$, with sizes 100 and 0,
and the first two ask levels are 1001 and 1002, with sizes 150 and 50.
By eq. (1.1.2, `eq.queueImbalance`)
$$I^2 = \frac{(100 + 0) - (150 + 50)}{(100 + 0) + (150 + 50)} = -\frac{100}{300} = -\frac13.$$

**Over occupied levels.**
`sorted(book.bids.items(), reverse=True)[:2]` takes the two highest bid prices, 1000 and 998: $b = 400$.
The asks give 1001 and 1002: $a = 200$.
The helper returns $(400 - 200)/600 = 1/3$.

A gap, at 999, falls within the first two bid levels,
so the two readings differ, Remark 1.2.4 (`remark.gridAgainstOccupied`).
The gap at 1003 falls outside the first two ask levels.

In [ ]:
grid = (100 + 0 - 150 - 50) / (100 + 0 + 150 + 50)
occupied = (400 - 200) / (400 + 200)
round(grid, 3), round(occupied, 3)

**The other choices.**
- (A) `-0.333 -0.333`: takes the helper to read grid levels; it skips 999.
- (B) `0.333 0.333`: takes `queue_imbalance` to count occupied levels;
  `bid_size(i)` reads the grid price $P^b - (i-1)$.
- (C) `0.333 -0.333`: the two readings exchanged,
  or equally the sign of eq. (1.1.2, `eq.queueImbalance`) inverted, ask minus bid.

## 1.3 · `question.microPriceTowardsTheThinSide`

Bids `{1000: 300}` and asks `{1002: 100}`.
`micro_price` against a mid that weights each price by its own size.

**Answer: (A).** `1001.5 1000.5`.

**The micro-price.**
$P^m = 1001$, $\phi = 2$ and $I^1 = (300 - 100)/400 = 1/2$ (F2),
so by eq. (1.1.3, `eq.microPrice`)
$$P^\mu = P^m + \frac{\phi}{2}\,I^1 = 1001 + 1 \cdot \tfrac12 = 1001.5.$$
The remark after eq. (1.1.3, `eq.microPrice`) writes it as the weighting
in which each price carries the size resting on the opposite side,
$$P^\mu = \frac{S^{b,1}P^a + S^{a,1}P^b}{S^{b,1} + S^{a,1}} = \frac{300 \cdot 1002 + 100 \cdot 1000}{400},$$
so that a bid-heavy book carries it towards the ask, the thin side.

**The size-weighted mid.**
`weighted_mid` gives each price its own size:
$(1000 \cdot 300 + 1002 \cdot 100)/400 = 400200/400 = 1000.5$,
which leans towards the side holding more, here the bid.

In [ ]:
imbalance = (300 - 100) / (300 + 100)
crossed = (300 * 1002 + 100 * 1000) / 400
own = (1000 * 300 + 1002 * 100) / 400
1001 + 2 / 2 * imbalance, crossed, own

**The other choices.**
- (B) `1000.5 1001.5`: the sizes of the two sides exchanged, 100 at the bid and 300 at the ask.
- (C) `1001.5 1001.5`: `weighted_mid` taken for the micro-price;
  it weights each price by its own size, not by the opposite one.
- (D) `1001.0 1001.0`: both taken for the mid-price, the sizes ignored;
  each differs from $P^m$ whenever $S^{b,1} \neq S^{a,1}$.

## 1.4 · `question.floatPricesSplitALevel`

A tick of 0.01, and three orders:
100 shares at 10.04 read from a feed, 50 shares at `1004 * tick`, and 70 shares at 10.03.
The stem gives the floating-point quotients.

**Answer: (B).** `{1003: 100, 1004: 50, 1002: 70}`.

`int` truncates towards zero, a fact of Python. With the quotients of the stem:
- `10.04 / 0.01` is `1003.9999999999999`, and the key is 1003;
- `1004 * 0.01` is `10.040000000000001`, its quotient `1004.0000000000001`, and the key is 1004;
- `10.03 / 0.01` is `1002.9999999999999`, and the key is 1002.

Each key is new, and the dictionary prints them in the order of insertion (F4).
The two orders at 10.04 land on two levels, and the order at 10.03 one tick below its price.

Inside the package prices are integer counts of ticks, converted to currency only at the boundary,
Section 1.2.3 (`sec.foldImplementation`).
The conversion from currency, `unito26.lob.messages.TickGrid.to_ticks`, uses `round`,
which gives `{1004: 150, 1003: 70}`.

In [ ]:
int(1003.9999999999999), int(1004.0000000000001), int(1002.9999999999999)

In [ ]:
round(1003.9999999999999), round(1004.0000000000001), round(1002.9999999999999)

**The other choices.**
- (A) `{1004: 150, 1003: 70}`: `int` read as `round`, the conversion of `TickGrid.to_ticks`.
- (C) `{1003: 150, 1002: 70}`: `1004 * tick` taken for the float 10.04;
  it is `10.040000000000001`, whose quotient truncates to 1004.
- (D) `{1003: 170, 1004: 50}`: 10.03 taken as exact, its quotient 1003;
  the stem gives `1002.9999999999999`.

## 1.5 · `question.recorderAliasesTheSide`

Bids `{1000: 100}` and asks `{1002: 120}`;
a limit buy of 50 at 1001, a market sell of 30, and a withdrawal of 120 at 1002 on the ask side.
`shared` appends `book.levels_map(BUY)`, and `copied` a `dict` of it.

**Answer: (C).** `{1000: 100, 1001: 20}`, then `{1000: 100, 1001: 50}`.

**The bids after each message** (F3).
- The buy at 1001 finds the best ask, 1002, above its limit, matches nothing and rests:
  `{1000: 100, 1001: 50}`.
- The market sell of 30 matches at the best bid, 1001: `{1000: 100, 1001: 20}`.
- The withdrawal subtracts 120 at 1002 on the ask side, and empties it; the bids are unchanged.

**What is recorded.**
`levels_map(BUY)` returns `book.bids` itself, not a copy, Section B.6 (`sec.stateAndInvariants`).
Every entry of `shared` is that one dictionary, `append` storing the object it is passed, a fact of Python;
the change the sell makes through `book.bids` is read through `shared[0]` (F4),
and `print` shows it as it is at the end, `{1000: 100, 1001: 20}`, with 1001 in its place (F4).
`dict(...)` builds a new dictionary (F4),
so `copied[0]` holds the bids after the first message, `{1000: 100, 1001: 50}`.

**The other choices.**
- (A) `{1000: 100, 1001: 50}` on both lines: takes `shared` to record the side as it is;
  it records a reference to the dictionary that the sell then changes.
- (B) `{1000: 100}`, then `{1000: 100, 1001: 50}`:
  sees the aliasing, and takes the withdrawal of 120 to act on the best bid, which it would empty;
  the withdrawal names the ask side and the price 1002.
- (D) `{1000: 100, 1001: 20}` on both lines: takes `dict(...)` for a view of the side; it is a copy.

## 1.6 · `question.sizeIsNotVolume`

Bids `{1000: 100, 999: 200}` and asks `{1002: 120}`;
a market sell of 30, a withdrawal of 50 at 1000, a limit buy of 40 at 1000, and a market sell of 70.
`drops` sums the decreases of $S^{b,1}$, and `traded` sums `last_traded_size`.

**Answer: (B).** `80 100`.

**The book after each message** (F3).

| message | bids | $S^{b,1}$ | decrease | traded |
| --- | --- | --- | --- | --- |
| | `{1000: 100, 999: 200}` | 100 | | |
| market sell of 30 | `{1000: 70, 999: 200}` | 70 | 30 | 30 |
| withdrawal of 50 at 1000 | `{1000: 20, 999: 200}` | 20 | 50 | 0 |
| limit buy of 40 at 1000 | `{1000: 60, 999: 200}` | 60 | 0 | 0 |
| market sell of 70 | `{999: 190}` | 190 | 0 | 70 |

The buy at 1000 finds the best ask, 1002, above its limit, and joins the queue at 1000.
The last sell takes the 60 at 1000, which `set_size` removes (F1), and 10 at 999.
The best bid moves to 999, whose size 190 exceeds 60, and `max(..., 0)` records no decrease.

So `drops` is $30 + 50 = 80$, and `traded` is $30 + 70 = 100$:
the sizes $q_M$ of the two market orders.
By Remark 1.1.3 (`remark.sizeVersusVolume`) the volume over a window is the sum of $q_M$
over the market orders within it, and no sequence of sizes determines it:
a level shrinks by cancellation as well as by execution.
Here, besides, the last sell moves the best bid to a larger queue, and `drops` records none of its 70 shares.

In [ ]:
at_1000 = 100 - 30 - 50 + 40
at_999 = 200 - (70 - at_1000)
at_1000, at_999, 30 + 50, 30 + 70

**The other choices.**
- (A) `100 100`: takes the decreases of the best size for the volume,
  counting the 70 of the last sell and not the withdrawal.
- (C) `140 100`: measures the last decrease at 1000, which falls from 60 to 0, $30 + 50 + 60$;
  `best_bid_size` reads the new best bid, 999.
- (D) `80 80`: takes `last_traded_size` for the fall of the best size,
  two quantities that Remark 1.1.3 (`remark.sizeVersusVolume`) separates.

## 1.7 · `question.paddingSentinelsInTheSpread`

Four rows of the first level, in ticks.
The bid of row 1 is padded with $-9999999999$ and the ask of row 2 with $9999999999$, counting rows from 0.
The filter keeps the rows with `BidPrice1 > 0`.

**Answer: (D).** `3` and a number near $3.3 \times 10^9$.

**The rows kept.**
The two sentinels have opposite signs.
`BidPrice1 > 0` drops row 1, whose bid is padded,
and keeps row 2, whose ask is: three rows.

**The mean.**
The spreads of the rows kept are $1002 - 1000 = 2$, $9999999999 - 1001 = 9999998998$ and $1002 - 1001 = 1$,
and their mean is $9999999001/3 \approx 3.33 \times 10^9$.
The verification notebook prints `3333333000.3333335`.

The orderbook file of LOBSTER pads a side with sentinels where it has fewer occupied prices than the depth of the file
(the subsection on LOBSTER in Section 1.2 (`sec.messageFiles`)).
At the first level a sentinel marks an empty side, where the spread of F2 is not defined.
A filter tests each side against its own sentinel, as `unito26.lob.lobster.touch_census` does;
it drops rows 1 and 2 and leaves the spreads 2 and 1, of mean 1.5.

In [ ]:
spreads = (1002 - 1000, 9999999999 - 1001, 1002 - 1001)
sum(spreads), sum(spreads) / 3, (2 + 1) / 2

**The other choices.**
- (A) `2` and `1.5`: takes the filter to drop both padded rows;
  it tests only the bid, and the ask sentinel is positive.
- (B) `3` and `1.5`: takes the padded ask for a missing value, which `mean` skips, a fact of pandas;
  it is an integer, and enters the mean.
- (C) `4` and a number near $3.3 \times 10^9$: takes `book[book.BidPrice1 > 0]` for `book.where(book.BidPrice1 > 0)`,
  which keeps every row and blanks row 1, a fact of pandas;
  `len` then counts four rows, and `mean` skips the blank spread.
  Boolean indexing drops the row.

## 1.8 · `question.levelsAreOneIndexed`

Bids `{1000: 100, 998: 150, 997: 50}`, with a gap at 999.
`size_a` reads the size at the price $P^b - (i-1)$, `size_b` at the price $P^b - i$.

**Answer: (A).** `[100, 0, 150]`, then `[0, 150, 50]`.

`size_at` reads a price, and returns zero where nothing rests (F1).
- `size_a` reads $1000 - (i - 1)$, the prices 1000, 999 and 998, of sizes 100, 0 and 150.
  These are $S^{b,i}$ for $i = 1, 2, 3$ (F2); 999 is a grid level holding nothing.
- `size_b` reads $1000 - i$, the prices 999, 998 and 997, of sizes 0, 150 and 50.
  It is $S^{b,i+1}$: a level counted from 0 read as one counted from 1.

**The other choices.**
- (B) `[100, 150, 50]`, then `[150, 50, 0]`: reads the $i$-th level as the $i$-th occupied price,
  skipping the gap at 999; both helpers read grid prices, Remark 1.2.4 (`remark.gridAgainstOccupied`).
- (C) `[100, 0, 150]` on both lines: reads `best_bid_price - i` as `best_bid_price - (i - 1)`.
- (D) `[0, 150, 50]`, then `[100, 0, 150]`: the two helpers exchanged,
  `size_b` taken for the one that counts from 1.

## 1.9 · `question.sizeAtOnADictAgainstAList`

One side with $\ell$ occupied levels, held as a dictionary (A) and as a list of pairs sorted by price (B),
queried at occupied prices, for $\ell = 10^3$ and $\ell = 10^4$.

**Answer: (C).**
A is faster; its time per query is about the same at both values of $\ell$, and that of B grows about tenfold.

**A.**
A Python dictionary is a hash table: `get` finds a key by its hash,
in a time that does not grow with $\ell$, on average.
This is a fact of Python.
Of the four operations of the storage, only `best_price` does work that grows with the book,
Section 1.2.4 (`sec.fasterBook`);
`size_at` is one of the other three, and it is this `get` (`unito26.lob.orderbook.AggregateBook.size_at`).

**B.**
The loop compares the prices in turn, from the first, and stops at the match.
A query at the $k$-th price makes $k$ comparisons,
and at a price drawn uniformly among the $\ell$ the mean is $(\ell + 1)/2$:
about 500 comparisons at $\ell = 10^3$ and about 5000 at $\ell = 10^4$,
against one lookup by hash for A.
The order of the list is not used.

A run of `01-representing-the-book` prints a growth near 1 for A and of the order of ten for B,
and B of the order of a thousand times slower than A at $\ell = 10^4$; the figures vary from run to run.

In [ ]:
(1_000 + 1) / 2, (10_000 + 1) / 2, (10_000 + 1) / (1_000 + 1)

**The other choices.**
- (A) B faster, its levels being sorted: the scan does not use the order.
  A search that did, by bisection, would make $O(\log \ell)$ comparisons, still growing with $\ell$.
- (B) Both grow tenfold, A faster by a constant factor: takes `get` to search the keys in turn;
  it goes to the key by its hash.
- (D) Neither changes noticeably: misses that the scan visits every price before the one it seeks.